# Inference & Submission — Prediction Track

Loads the trained seq2seq checkpoints, rebuilds the same feature pipeline, and emits a Kaggle-ready `submission.csv`. Point this notebook at the dataset that holds your exported weights + vocab from the training notebook.

## 1. Locate code + data
Works both on Kaggle (where the competition dataset lives under `/kaggle/input`) and locally when you keep the official zip file next to the repo.

In [ ]:
import json
import os
from pathlib import Path
import sys

import pandas as pd
import torch
from torch.utils.data import DataLoader

PROJECT_ROOT = Path.cwd().resolve()


def _gather_src_candidates(base: Path) -> list[Path]:
    rel_paths = (
        'src',
        'prediction/src',
        'workspace/prediction/src',
    )
    out = []
    for rel in rel_paths:
        out.append((base / rel).resolve())
    return out


def resolve_src_dir() -> Path:
    candidates = []
    seen = set()

    def add(path: Path):
        path = Path(path)
        try:
            path = path.resolve()
        except Exception:
            pass
        if path in seen:
            return
        seen.add(path)
        candidates.append(path)

    env_src = os.getenv('BDB2026_SRC')
    if env_src:
        add(Path(env_src))

    search_roots = [PROJECT_ROOT] + list(PROJECT_ROOT.parents[:5])
    for root in search_roots:
        for cand in _gather_src_candidates(root):
            add(cand)

    kaggle_work = Path('/kaggle/working')
    if kaggle_work.exists():
        for cand in _gather_src_candidates(kaggle_work):
            add(cand)

    kaggle_input = Path('/kaggle/input')
    if kaggle_input.exists():
        for child in kaggle_input.iterdir():
            cand = child / 'src'
            add(cand)
            cand_pred = child / 'prediction' / 'src'
            add(cand_pred)

    for cand in candidates:
        if cand.exists():
            path_str = str(cand)
            if path_str not in sys.path:
                sys.path.append(path_str)
            return cand

    raise FileNotFoundError(
        'Could not locate prediction/src directory. Set BDB2026_SRC or mount the repo next to this notebook.'
    )


SRC_DIR = resolve_src_dir()
print('Using src dir:', SRC_DIR)

## 2. Load configs, vocab, and checkpoints
By default we look under `../outputs/prediction/checkpoints`. On Kaggle you should point `BDB2026_WEIGHTS` to the path where you mounted your dataset of saved weights.

In [ ]:
import os
from pathlib import Path

if 'PROJECT_ROOT' not in globals():
    PROJECT_ROOT = Path.cwd().resolve()

def _maybe_path(value):
    if not value:
        return None
    return Path(value).expanduser()

def _ensure_data_source():
    global DATA_ROOT, DATA_ZIP
    if 'DATA_ROOT' in globals() and 'DATA_ZIP' in globals():
        if DATA_ROOT is not None or DATA_ZIP is not None:
            return
    data_candidates = [
        _maybe_path(os.getenv('BDB2026_DATA')),
        Path('/kaggle/input/nfl-big-data-bowl-2026'),
        PROJECT_ROOT / '../input/nfl-big-data-bowl-2026',
        PROJECT_ROOT / '../../input/nfl-big-data-bowl-2026',
    ]
    for cand in data_candidates:
        if cand and cand.exists():
            DATA_ROOT = cand.resolve()
            DATA_ZIP = None
            return
    zip_candidates = [
        _maybe_path(os.getenv('BDB2026_ZIP')),
        PROJECT_ROOT / '../nfl-big-data-bowl-2026-prediction.zip',
        PROJECT_ROOT / '../../nfl-big-data-bowl-2026-prediction.zip',
        PROJECT_ROOT / 'nfl-big-data-bowl-2026-prediction.zip',
    ]
    for cand in zip_candidates:
        if cand and cand.exists():
            DATA_ROOT = None
            DATA_ZIP = cand.resolve()
            return
    raise FileNotFoundError('Could not locate data directory or zip archive. Set BDB2026_DATA/BDB2026_ZIP.')

_ensure_data_source()
print('Using data root:', DATA_ROOT)
print('Using data zip:', DATA_ZIP)

In [ ]:
from bdb2026.config import ExperimentConfig
from bdb2026.data import CompetitionData, build_test_records, load_vocabulary
from bdb2026.dataset import TrajectoryDataset, collate_fn
from bdb2026.model import TrajectoryModel
from bdb2026.inference import run_inference, predictions_to_frame, persistence_baseline

ARTIFACT_FILES = ['prgt_baseline.pt', 'vocab.json', 'config.json']


def _gather_weight_dirs() -> list[Path]:
    candidates: list[Path] = []

    def add(path_like):
        if not path_like:
            return
        path = Path(path_like).expanduser()
        if path.is_file():
            path = path.parent
        try:
            path = path.resolve()
        except Exception:
            pass
        if path not in candidates:
            candidates.append(path)

    add(os.getenv('BDB2026_WEIGHTS', ''))
    add(PROJECT_ROOT / 'outputs/prediction/checkpoints')
    add(PROJECT_ROOT / '../outputs/prediction/checkpoints')
    add(PROJECT_ROOT / '../../outputs/prediction/checkpoints')
    add(PROJECT_ROOT / 'checkpoints')
    add(PROJECT_ROOT / '../checkpoints')

    kaggle_work = Path('/kaggle/working')
    if kaggle_work.exists():
        add(kaggle_work / 'outputs/prediction/checkpoints')
        add(kaggle_work / 'prediction/outputs/prediction/checkpoints')

    kaggle_input = Path('/kaggle/input')
    if kaggle_input.exists():
        for dataset_dir in kaggle_input.iterdir():
            if dataset_dir.is_dir():
                add(dataset_dir)
                add(dataset_dir / 'outputs/prediction/checkpoints')
                add(dataset_dir / 'prediction/outputs/prediction/checkpoints')
    return candidates


def _dir_has_artifacts(path: Path) -> bool:
    return all((path / name).exists() for name in ARTIFACT_FILES)


def resolve_weights_dir() -> Path:
    for cand in _gather_weight_dirs():
        if cand and cand.exists() and _dir_has_artifacts(cand):
            return cand

    search_roots = [
        PROJECT_ROOT,
        PROJECT_ROOT / '..',
        PROJECT_ROOT / '../..',
        Path('/kaggle/working'),
        Path('/kaggle/input'),
    ]
    seen = set()
    for root in search_roots:
        if not root or not root.exists():
            continue
        root = root.resolve()
        if root in seen:
            continue
        seen.add(root)
        try:
            matches = list(root.rglob(ARTIFACT_FILES[0]))
        except Exception:
            continue
        for match in matches:
            candidate_dir = match.parent
            if _dir_has_artifacts(candidate_dir):
                return candidate_dir

    raise FileNotFoundError(
        'Could not locate checkpoint artifacts. Provide prgt_baseline.pt, vocab.json, and config.json via BDB2026_WEIGHTS or place them in outputs/prediction/checkpoints.'
    )


try:
    weights_dir = resolve_weights_dir()
    checkpoint_path = weights_dir / ARTIFACT_FILES[0]
    vocab_path = weights_dir / ARTIFACT_FILES[1]
    config_path = weights_dir / ARTIFACT_FILES[2]
    print('Checkpoint directory:', weights_dir)
    config = ExperimentConfig.from_dict(json.loads(config_path.read_text()))
    vocab = load_vocabulary(vocab_path)
    USE_MODEL = True
except FileNotFoundError:
    print('No checkpoint artifacts found; using persistence baseline (repeats last observed position).')
    weights_dir = None
    config = ExperimentConfig()
    config.model.decoder_steps = config.data.decoder_frames
    vocab = None
    USE_MODEL = False


## 3. Build test samples + dataloader

In [ ]:
competition_data = CompetitionData(data_root=DATA_ROOT, zip_path=DATA_ZIP)

test_samples, vocab = build_test_records(
    competition_data,
    encoder_frames=config.data.encoder_frames,
    decoder_frames=config.data.decoder_frames,
    vocab=vocab,
)
print(f'Test samples: {len(test_samples):,}')

if USE_MODEL:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    test_ds = TrajectoryDataset(test_samples, augment=False)
    test_loader = DataLoader(
        test_ds,
        batch_size=config.optim.val_batch_size,
        shuffle=False,
        num_workers=config.optim.num_workers,
        pin_memory=device.type == 'cuda',
        collate_fn=collate_fn,
    )

    feature_dim = test_samples[0].context_features.shape[-1]
    static_dim = test_samples[0].static_features.shape[-1]
    config.model.feature_dim = feature_dim
    config.model.static_dim = static_dim
    config.model.decoder_steps = config.data.decoder_frames

    vocab_sizes = {name: len(v) for name, v in vocab.items()}
    checkpoint_files = sorted(weights_dir.glob('*.pt')) or [checkpoint_path]
else:
    test_loader = None
    device = torch.device('cpu')
    checkpoint_files = []


## 4. Run inference & build submission

In [ ]:
from bdb2026.data import mirror_sample

if USE_MODEL:
    if not checkpoint_files:
        raise FileNotFoundError('No checkpoints available for inference.')

    def build_loader(sample_list):
        ds = TrajectoryDataset(sample_list, augment=False)
        return DataLoader(
            ds,
            batch_size=config.optim.val_batch_size,
            shuffle=False,
            num_workers=config.optim.num_workers,
            pin_memory=device.type == 'cuda',
            collate_fn=collate_fn,
        )

    def run_with_checkpoints(sample_list):
        loader = build_loader(sample_list)
        ensemble_preds = None
        passes = 0
        for ckpt in checkpoint_files:
            print(f'Running checkpoint {ckpt.name}')
            model = TrajectoryModel(config.model, feature_dim, static_dim, vocab_sizes).to(device)
            checkpoint = torch.load(ckpt, map_location=device)
            model.load_state_dict(checkpoint['model_state'])
            preds = run_inference(model, loader, device)
            if sample_list and sample_list[0].mirror:
                for rec in preds:
                    rec['predictions'][:, 1] *= -1
                    rec['mirror'] = False
            if ensemble_preds is None:
                ensemble_preds = preds
            else:
                for base, extra in zip(ensemble_preds, preds):
                    base['predictions'] += extra['predictions']
            passes += 1
        return ensemble_preds, passes

    base_preds, total_passes = run_with_checkpoints(test_samples)
    mirror_samples = [mirror_sample(sample) for sample in test_samples]
    mirror_preds, mirror_passes = run_with_checkpoints(mirror_samples)
    if mirror_preds is not None and mirror_passes > 0:
        total_passes += mirror_passes
        for base, extra in zip(base_preds, mirror_preds):
            base['predictions'] += extra['predictions']
    for rec in base_preds:
        rec['predictions'] /= max(total_passes, 1)
    predictions = base_preds
else:
    predictions = persistence_baseline(test_samples)

submission = predictions_to_frame(predictions)
submission_path = Path('submission.csv')
submission.to_csv(submission_path, index=False)
print('Wrote submission with', len(submission), 'rows to', submission_path)
submission.head()

## 5. Checklist before Kaggle submission
- Double-check the checkpoint/vocab/config trio came from the same training run.
- When creating a Kaggle Notebook, add your weights dataset plus the official competition dataset as inputs.
- Keep `submission.csv` under ~150 MB to satisfy the limit (this baseline outputs ~60-70 MB).
- Feel free to chain ensembling or smoothing here once you have multiple checkpoints.